# Exploración aislada de inventory.json

Mi objetivo es entender estructura, granularidad, calidad, cobertura y riesgos para rotación, quiebres y margen. Analizo únicamente `datos/inventory.json`: no leo ni cruzo sales, e-commerce ni tipos de cambio. Las relaciones entre bloques de este mismo JSON son controles internos, no conciliaciones entre fuentes. Conservo los originales y separo las columnas derivadas. No hay respuestas del cliente implícitas en los nombres de campos.

## Documentación relacionada

- [Resultados de inventory](../docs/02_exploracion/resultados_inventory.md).

In [1]:
# Localizo la raíz y preparo tablas con leyendas compartidas por el notebook y su informe.
from pathlib import Path
import json
import hashlib
import copy
from collections import Counter
from datetime import date, timedelta
from decimal import Decimal
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, HTML

raiz = Path.cwd() if (Path.cwd() / "datos/inventory.json").exists() else Path.cwd().parent
ruta = raiz / "datos/inventory.json"
tablas = {}
def mostrar(nombre, tabla, leyenda):
    tablas[nombre] = (tabla.copy(deep=True), leyenda)
    display(Markdown("**Leyenda:** " + leyenda + " No muestro el índice de pandas: no es una clave de origen."))
    display(HTML(tabla.to_html(index=False)))
def medidas(valores):
    return pd.DataFrame({"Medida": list(valores), "Resultado": pd.Series(list(valores.values()), dtype=object)})

## 1. Lectura e identificación de la versión

Leo UTF-8 y conservo tipos JSON, nulos literales y textos como `N/A`. Compruebo también claves repetidas dentro de objetos JSON: una lectura ordinaria podría ocultarlas conservando solo la última. La fecha de generación de metadata no amplía la ventana de snapshots y carece de zona horaria explícita.

In [2]:
# Identifico los bytes leídos y detecto claves JSON repetidas antes de aplanar las colecciones.
contenido = ruta.read_bytes()
huella = hashlib.sha256(contenido).hexdigest()
claves_repetidas_json = []
def objeto_controlado(pares):
    repetidas = [k for k, n in Counter(k for k, v in pares).items() if n > 1]
    claves_repetidas_json.extend(repetidas)
    return dict(pares)
datos = json.loads(contenido.decode("utf-8"), object_pairs_hook=objeto_controlado)
original = copy.deepcopy(datos)
tiendas = pd.DataFrame(datos["tiendas_info"])
mapeos = pd.DataFrame(datos["sku_mappings"])
productos = pd.DataFrame(datos["catalogo"]["productos"])
costos_filas = [dict(sku_erp=p["sku_erp"], **c) for p in datos["catalogo"]["productos"] for c in p["cost_history"]]
costos = pd.DataFrame(costos_filas)
snapshots_original = pd.DataFrame(datos["snapshots"])
snapshots = snapshots_original.copy(deep=True)
mostrar("fuente", medidas({"Archivo": "datos/inventory.json", "Bytes": len(contenido), "SHA-256": huella, "Generado según metadata": datos["metadata"]["generado"], "Sistema declarado": datos["metadata"]["fuente"], "Descripción declarada": datos["metadata"]["descripcion"], "Claves repetidas en objetos JSON": len(claves_repetidas_json)}), "Identifico la extracción exacta; bytes es tamaño físico y metadata es una declaración del archivo, no verificación del ERP.")

**Leyenda:** Identifico la extracción exacta; bytes es tamaño físico y metadata es una declaración del archivo, no verificación del ERP. No muestro el índice de pandas: no es una clave de origen.

Medida,Resultado
Archivo,datos/inventory.json
Bytes,31671112
SHA-256,2355c6cab9ff00a707b5505a464a8f7e1a9bf5a2aa6c7a2b6b008f5299c8eada
Generado según metadata,2026-04-23T08:22:39.344087
Sistema declarado,ERP CaféNorte v2.3 (legacy)
Descripción declarada,Snapshots diarios de inventario. No incluye movimientos transaccionales.
Claves repetidas en objetos JSON,0


## 2. Esquema, diccionario y granularidad

Distingo tiendas, equivalencias, productos, vigencias de costo y snapshots. Mi clave candidata de snapshot es fecha–tienda–SKU ERP; la valido antes de medir continuidad. El historial de costos pertenece al producto y no contiene tienda. Los nombres de campos no explicitan unidad física, base fiscal, método de costo ni hora del saldo.

In [3]:
# Perfilo todas las colecciones, separando campos ausentes de null literal y tipos de origen.
colecciones = {"tiendas": datos["tiendas_info"], "mapeos": datos["sku_mappings"], "productos": datos["catalogo"]["productos"], "costos": costos_filas, "snapshots": datos["snapshots"]}
esquema = []
for bloque, filas in colecciones.items():
    campos = sorted(set().union(*(r.keys() for r in filas)))
    for campo in campos:
        valores = [r[campo] for r in filas if campo in r]
        esquema.append({"Bloque": bloque, "Campo": campo, "Registros": len(filas), "Tipos JSON leídos": ", ".join(sorted({type(v).__name__ for v in valores})), "Campo ausente": sum(campo not in r for r in filas), "Null literal": sum(v is None for v in valores), "Texto vacío": sum(isinstance(v, str) and not v.strip() for v in valores), "Espacios extremos": sum(isinstance(v, str) and v != v.strip() for v in valores)})
mostrar("esquema", pd.DataFrame(esquema), "Reviso todos los registros por bloque y campo; las cuatro últimas columnas cuentan registros afectados. list identifica el historial anidado; costos añade sku_erp como vínculo derivado al padre.")
diccionario = [
    ("metadata", "generado, fuente, descripcion", "Marca de generación sin zona explícita; sistema y alcance declarados."),
    ("tiendas_info", "tienda_id, ciudad, region, timezone", "Identidad y atributos de tienda; timezone no confirma la hora del POS."),
    ("sku_mappings", "sku_pos, sku_erp, handle", "Equivalencias declaradas; null no es una clave y no se enlaza con otro null."),
    ("catalogo.productos", "sku_erp, nombre, categoria", "Identidad ERP y descripción; un nombre compartido no demuestra mismo producto."),
    ("cost_history", "fecha_vigencia, costo_mxn, proveedor", "Historial por producto; MXN según nombre del campo, sin unidad, base fiscal ni política explícitas."),
    ("snapshots", "fecha, tienda_id, sku_erp, cantidad_en_stock", "Saldo diario observado; no son movimientos. N/A conserva estado desconocido; sin unidad ni hora de captura explícitas.")]
mostrar("diccionario", pd.DataFrame(diccionario, columns=["Bloque", "Campos originales", "Interpretación y límite"]), "Diccionario de lectura de los campos del JSON; no constituye un contrato de negocio confirmado.")

**Leyenda:** Reviso todos los registros por bloque y campo; las cuatro últimas columnas cuentan registros afectados. list identifica el historial anidado; costos añade sku_erp como vínculo derivado al padre. No muestro el índice de pandas: no es una clave de origen.

Bloque,Campo,Registros,Tipos JSON leídos,Campo ausente,Null literal,Texto vacío,Espacios extremos
tiendas,ciudad,40,str,0,0,0,0
tiendas,region,40,str,0,0,0,0
tiendas,tienda_id,40,str,0,0,0,0
tiendas,timezone,40,str,0,0,0,0
mapeos,handle,65,"NoneType, str",0,38,0,0
mapeos,sku_erp,65,"NoneType, str",0,5,0,0
mapeos,sku_pos,65,str,0,0,0,0
productos,categoria,70,str,0,0,0,0
productos,cost_history,70,list,0,0,0,0
productos,nombre,70,str,0,0,0,0


**Leyenda:** Diccionario de lectura de los campos del JSON; no constituye un contrato de negocio confirmado. No muestro el índice de pandas: no es una clave de origen.

Bloque,Campos originales,Interpretación y límite
metadata,"generado, fuente, descripcion",Marca de generación sin zona explícita; sistema y alcance declarados.
tiendas_info,"tienda_id, ciudad, region, timezone",Identidad y atributos de tienda; timezone no confirma la hora del POS.
sku_mappings,"sku_pos, sku_erp, handle",Equivalencias declaradas; null no es una clave y no se enlaza con otro null.
catalogo.productos,"sku_erp, nombre, categoria",Identidad ERP y descripción; un nombre compartido no demuestra mismo producto.
cost_history,"fecha_vigencia, costo_mxn, proveedor","Historial por producto; MXN según nombre del campo, sin unidad, base fiscal ni política explícitas."
snapshots,"fecha, tienda_id, sku_erp, cantidad_en_stock",Saldo diario observado; no son movimientos. N/A conserva estado desconocido; sin unidad ni hora de captura explícitas.


## 3. Calidad de claves, referencias y saldo

Conservo `cantidad_en_stock` intacta. Derivo `stock_num` y `estado_stock`: separo null literal, texto `N/A`, no numérico, no finito, negativo, cero, positivo y booleano. Un cero observado no equivale a falta de fila. Compruebo referencias solo contra las tiendas y productos del propio JSON. Los patrones de identificadores son controles sintácticos y no prueban identidad comercial.

### Columnas que creo para analizar los snapshots

Mantengo `cantidad_en_stock` y `fecha` tal como llegaron en el JSON. Añado las siguientes columnas únicamente en la tabla de trabajo; no las escribo en la fuente.

**Leyenda:** describo las tres columnas derivadas de esta etapa, su campo de origen y su uso. `NaN` representa un valor numérico ausente en pandas y `NaT` una fecha ausente; ninguno equivale a cero. El archivo no explicita la unidad física de existencias.

| Columna derivada | Campo de origen | Qué significa y cómo la obtengo | Para qué la uso |
|---|---|---|---|
| `stock_num` | `cantidad_en_stock` | Es la versión convertida a número con `pd.to_numeric(..., errors="coerce")`. Conservo los valores interpretables como números; lo que no se puede convertir queda como `NaN`. La conversión por sí sola no certifica que el saldo sea válido. | Calculo estadísticas de saldos conocidos y detecto ceros; consulto `estado_stock` para distinguir el motivo y las características del valor original. |
| `estado_stock` | `cantidad_en_stock` | Es una etiqueta que clasifica el valor original: null literal, N/A, otro no numérico, no finito, booleano, negativo, cero o positivo. | Distingo un saldo desconocido de un cero y cuento cada condición sin perder la evidencia original. |
| `fecha_dt` | `fecha` | Es la fecha interpretada con el formato `AAAA-MM-DD`, mediante `pd.to_datetime(..., errors="coerce")`. Si no se puede interpretar queda como `NaT`; no añado hora de captura ni zona horaria. | Ordeno los snapshots, comparo días y construyo calendarios y rachas consecutivas. |

Por ejemplo, si el saldo original es `31`, obtengo `stock_num = 31` y `estado_stock = "Positivo"`; si es `0`, obtengo `stock_num = 0` y `estado_stock = "Cero"`. Si el original es el texto `"N/A"`, obtengo `stock_num = NaN` y `estado_stock = "N/A"`. Son ejemplos ilustrativos de la conversión, no nuevos resultados calculados ni filas seleccionadas del archivo.

### Significado de las etiquetas de estado_stock

**Leyenda:** explico todas las categorías previstas por el control, aunque alguna no aparezca en esta extracción. Las etiquetas describen el valor de una fila existente, no la ausencia de un snapshot.

| Etiqueta | Cómo interpreto la clasificación |
|---|---|
| `Null literal` | El valor original es `null` en el JSON; no contiene una cantidad informada. |
| `N/A` | El original contiene exactamente el texto `"N/A"`. Lo clasifico como saldo desconocido; no lo convierto en cero. |
| `No numérico distinto de N/A` | El original no se puede interpretar como número y no es el texto exacto `"N/A"`. |
| `No finito` | El valor se interpreta como número, pero es infinito o no representa un número finito. |
| `Booleano` | El original es `true` o `false`. Lo separo de las cantidades, aunque Python pueda tratar booleanos como números. |
| `Negativo` | El valor numérico finito es menor que cero. La etiqueta no confirma por sí sola un error del ERP. |
| `Cero` | El valor numérico finito es exactamente cero: existe una fila que informa ese saldo. |
| `Positivo` | El valor numérico finito es mayor que cero. |

Si no existe una fila para cierta fecha, tienda y producto, no hay valor que clasificar: no creo automáticamente un `estado_stock` ni un saldo cero para esa ausencia. Los estados de calidad no equivalen a respuestas del negocio; por ejemplo, un cero diario no demuestra disponibilidad nula durante todo el día.

In [4]:
# Compruebo claves y clasifico existencias sin reemplazar desconocidos por cero.
snapshots["fecha_dt"] = pd.to_datetime(snapshots["fecha"], format="%Y-%m-%d", errors="coerce")
snapshots["stock_num"] = pd.to_numeric(snapshots["cantidad_en_stock"], errors="coerce")
def clasificar(v):
    if v is None: return "Null literal"
    if isinstance(v, bool): return "Booleano"
    if v == "N/A": return "N/A"
    try: n = float(v)
    except (ValueError, TypeError): return "No numérico distinto de N/A"
    if not np.isfinite(n): return "No finito"
    return "Negativo" if n < 0 else "Cero" if n == 0 else "Positivo"
snapshots["estado_stock"] = snapshots["cantidad_en_stock"].map(clasificar)
claves = ["tienda_id", "sku_erp", "fecha_dt"]
controles = {"Snapshots": len(snapshots), "Duplicados exactos adicionales": int(snapshots_original.duplicated().sum()), "Claves tienda–SKU–fecha adicionales": int(snapshots.duplicated(claves).sum()), "Fechas sin interpretar": int(snapshots["fecha_dt"].isna().sum()), "Fechas fuera de formato ISO diario": int((~snapshots["fecha"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")).sum()), "Tiendas duplicadas adicionales": int(tiendas.duplicated("tienda_id").sum()), "Productos ERP duplicados adicionales": int(productos.duplicated("sku_erp").sum()), "Snapshots con tienda fuera del catálogo interno": int((~snapshots["tienda_id"].isin(tiendas["tienda_id"])).sum()), "Snapshots con SKU fuera del catálogo interno": int((~snapshots["sku_erp"].isin(productos["sku_erp"])).sum()), "SKU ERP de mapeos fuera del catálogo interno": int((~mapeos["sku_erp"].dropna().isin(productos["sku_erp"])).sum()), "Stock numérico finito no entero": int((np.isfinite(snapshots["stock_num"]) & snapshots["stock_num"].mod(1).ne(0)).sum()), "Tienda con patrón distinto de T y tres dígitos": int((~tiendas["tienda_id"].str.fullmatch(r"T[0-9]{3}")).sum()), "ERP con patrón distinto del observado": int((~productos["sku_erp"].str.fullmatch(r"ERP-PROV-MX-[0-9]{3}-[A-Z]")).sum()), "POS con patrón distinto de CN y cinco dígitos": int((~mapeos["sku_pos"].str.fullmatch(r"CN-[0-9]{5}")).sum())}
mostrar("calidad", medidas(controles), "Controles sobre la extracción completa; duplicados adicionales excluye la primera aparición de cada clave o fila. No elimino registros ni los declaro errores del ERP por ausencia de un catálogo externo.")
estados = ["Null literal", "N/A", "No numérico distinto de N/A", "No finito", "Booleano", "Negativo", "Cero", "Positivo"]
estado = snapshots["estado_stock"].value_counts().reindex(estados, fill_value=0).rename_axis("Estado del saldo").reset_index(name="Registros")
estado["Porcentaje de snapshots"] = (100 * estado["Registros"] / len(snapshots)).round(4)
mostrar("estados", estado, "Cuento todos los snapshots por estado exclusivo; porcentaje sobre todas las filas, no sobre productos o unidades. Mantengo categorías con cero casos para distinguirlas de controles omitidos.")

**Leyenda:** Controles sobre la extracción completa; duplicados adicionales excluye la primera aparición de cada clave o fila. No elimino registros ni los declaro errores del ERP por ausencia de un catálogo externo. No muestro el índice de pandas: no es una clave de origen.

Medida,Resultado
Snapshots,230776
Duplicados exactos adicionales,0
Claves tienda–SKU–fecha adicionales,0
Fechas sin interpretar,0
Fechas fuera de formato ISO diario,0
Tiendas duplicadas adicionales,0
Productos ERP duplicados adicionales,0
Snapshots con tienda fuera del catálogo interno,0
Snapshots con SKU fuera del catálogo interno,0
SKU ERP de mapeos fuera del catálogo interno,0


**Leyenda:** Cuento todos los snapshots por estado exclusivo; porcentaje sobre todas las filas, no sobre productos o unidades. Mantengo categorías con cero casos para distinguirlas de controles omitidos. No muestro el índice de pandas: no es una clave de origen.

Estado del saldo,Registros,Porcentaje de snapshots
Null literal,0,0.0000
N/A,4417,1.9140
No numérico distinto de N/A,0,0.0000
No finito,0,0.0000
Booleano,0,0.0000
Negativo,0,0.0000
Cero,11726,5.0811
Positivo,214633,93.0049


## 4. Cobertura temporal y surtido observado

Comparo el calendario diario inclusivo entre las fechas extremas con cada par tienda–SKU observado. Esa población no representa necesariamente el surtido autorizado. El producto cartesiano de tiendas y productos es solo un escenario de cobertura: sus ausencias no son quiebres ni fallos confirmados. Distingo integridad de filas de integridad de saldos numéricos; una fila `N/A` cuenta como presente en calendario, pero su saldo sigue desconocido.

In [5]:
# Mido continuidad de los pares observados y separo el calendario de la disponibilidad de cantidades.
inicio, fin = snapshots["fecha_dt"].min(), snapshots["fecha_dt"].max()
calendario = pd.date_range(inicio, fin, freq="D")
pares = snapshots.groupby(["tienda_id", "sku_erp"]).agg(filas=("fecha", "size"), dias=("fecha_dt", "nunique"), dias_con_saldo=("stock_num", "count"))
resumen_cobertura = {"Primera fecha snapshot": str(inicio.date()), "Última fecha snapshot": str(fin.date()), "Días del calendario inclusivo": len(calendario), "Fechas distintas observadas": snapshots["fecha_dt"].nunique(), "Tiendas observadas": snapshots["tienda_id"].nunique(), "SKU ERP observados": snapshots["sku_erp"].nunique(), "Pares tienda–SKU observados": len(pares), "Pares con todos los días observados": int(pares["dias"].eq(len(calendario)).sum()), "Días-par ausentes dentro de pares observados": int((len(calendario)-pares["dias"]).sum()), "Pares con al menos un saldo desconocido": int(pares["dias_con_saldo"].lt(pares["dias"]).sum()), "Pares cartesianos de tiendas y catálogo": len(tiendas)*len(productos), "Pares cartesianos nunca observados": len(tiendas)*len(productos)-len(pares)}
mostrar("cobertura", medidas(resumen_cobertura), "Cobertura del calendario completo; día-par es una fecha para una tienda y SKU. El escenario cartesiano no es surtido confirmado; no asigno saldo a sus pares ausentes.")
por_tienda = snapshots.groupby("tienda_id").agg(SKU_observados=("sku_erp", "nunique"), Dias_observados=("fecha_dt", "nunique"), Snapshots=("fecha", "size"), Saldos_desconocidos=("stock_num", lambda s: s.isna().sum()), Ceros_observados=("stock_num", lambda s: s.eq(0).sum())).reset_index()
por_tienda["Porcentaje_desconocido"] = (100*por_tienda["Saldos_desconocidos"]/por_tienda["Snapshots"]).round(3)
mostrar("tiendas", por_tienda, "Todas las tiendas, ordenadas por tienda_id; SKU y días son valores distintos, snapshots y saldos son registros. El porcentaje desconocido usa las filas de cada tienda; no presupongo que todas deban vender el catálogo completo.")
por_mes = snapshots.assign(mes=snapshots["fecha_dt"].dt.strftime("%Y-%m")).groupby("mes").agg(Dias=("fecha_dt", "nunique"), Snapshots=("fecha", "size"), Desconocidos=("stock_num", lambda s: s.isna().sum()), Ceros=("stock_num", lambda s: s.eq(0).sum())).reset_index()
por_mes["Porcentaje_desconocido"] = (100*por_mes["Desconocidos"]/por_mes["Snapshots"]).round(3)
mostrar("meses", por_mes, "Todos los meses observados en orden cronológico; desconocidos y ceros cuentan filas, no días completos ni unidades. El denominador porcentual son snapshots de cada mes.")
mostrar("zonas", tiendas.groupby(["region", "timezone"]).agg(Tiendas=("tienda_id", "nunique")).reset_index(), "Agrupo el catálogo interno de tiendas por región y zona declarada; no interpreto fechas diarias como instantes ni valido la zona del POS.")

**Leyenda:** Cobertura del calendario completo; día-par es una fecha para una tienda y SKU. El escenario cartesiano no es surtido confirmado; no asigno saldo a sus pares ausentes. No muestro el índice de pandas: no es una clave de origen.

Medida,Resultado
Primera fecha snapshot,2025-10-01
Última fecha snapshot,2026-03-31
Días del calendario inclusivo,182
Fechas distintas observadas,182
Tiendas observadas,40
SKU ERP observados,70
Pares tienda–SKU observados,1268
Pares con todos los días observados,1268
Días-par ausentes dentro de pares observados,0
Pares con al menos un saldo desconocido,1232


**Leyenda:** Todas las tiendas, ordenadas por tienda_id; SKU y días son valores distintos, snapshots y saldos son registros. El porcentaje desconocido usa las filas de cada tienda; no presupongo que todas deban vender el catálogo completo. No muestro el índice de pandas: no es una clave de origen.

tienda_id,SKU_observados,Dias_observados,Snapshots,Saldos_desconocidos,Ceros_observados,Porcentaje_desconocido
T001,40,182,7280,131,393,1.799
T002,26,182,4732,92,250,1.944
T003,34,182,6188,117,319,1.891
T004,38,182,6916,119,348,1.721
T005,36,182,6552,116,343,1.770
T006,40,182,7280,150,400,2.060
T007,28,182,5096,100,228,1.962
T008,25,182,4550,105,220,2.308
T009,31,182,5642,108,287,1.914
T010,30,182,5460,125,264,2.289


**Leyenda:** Todos los meses observados en orden cronológico; desconocidos y ceros cuentan filas, no días completos ni unidades. El denominador porcentual son snapshots de cada mes. No muestro el índice de pandas: no es una clave de origen.

mes,Dias,Snapshots,Desconocidos,Ceros,Porcentaje_desconocido
2025-10,31,39308,743,2022,1.890
2025-11,30,38040,753,1912,1.979
2025-12,31,39308,763,2023,1.941
2026-01,31,39308,745,1993,1.895
2026-02,28,35504,662,1824,1.865
2026-03,31,39308,751,1952,1.911


**Leyenda:** Agrupo el catálogo interno de tiendas por región y zona declarada; no interpreto fechas diarias como instantes ni valido la zona del POS. No muestro el índice de pandas: no es una clave de origen.

region,timezone,Tiendas
centro,America/Chihuahua,3
centro,America/Mexico_City,15
centro,America/Monterrey,3
frontera,America/Matamoros,4
frontera,America/Ojinaga,2
frontera,America/Tijuana,4
noroeste,America/Hermosillo,3
sureste,America/Cancun,3
sureste,America/Merida,3


## 5. Distribución de existencias

Mi objetivo aquí es responder: **¿qué cantidades de inventario aparecen habitualmente en los snapshots y cuánto varían?** Para eso reviso los valores de `stock_num` de todos los productos, tiendas y días del archivo.

Cada observación es **el saldo de un producto en una tienda para un día**. Si ese mismo producto y tienda aparecen en varias fechas, cada fecha aporta una observación. Incluyo los saldos cero porque son cantidades informadas; excluyo los `N/A` porque su cantidad es desconocida.

### Qué hago en el código

1. **Resumo todos los saldos conocidos.** Con `describe(...)` calculo cuántas observaciones tienen número, su promedio, dispersión, percentiles y extremos. Estos resultados aparecen en la primera tabla.
2. **Repito parte del resumen por categoría.** Con `merge(...)` añado a cada snapshot la categoría de su producto, usando el SKU ERP del catálogo de este mismo JSON. Después agrupo por categoría para contar productos y registros y describir sus saldos.
3. **Busco dónde aparece el saldo más alto.** Selecciono las filas cuyo saldo coincide con el máximo global para identificar fecha, tienda y producto. Si hay varios empates, muestro hasta diez, ordenados por fecha, tienda y SKU.

### Cómo leo la primera tabla

**Leyenda:** explico las medidas de la tabla de saldos conocidos que aparece después del código. Las observaciones cuentan registros; las demás medidas describen cantidades de stock, sin unidad física explícita en el archivo. Esta tabla explica conceptos y no añade nuevos resultados.

| Medida | Qué significa |
|---|---|
| Observaciones | Cantidad de snapshots que tienen un saldo numérico conocido. No es cantidad de productos ni suma de existencias. |
| Media | Promedio de los saldos conocidos: sumo sus valores y divido entre la cantidad de observaciones. Cada snapshot tiene el mismo peso. |
| Desviación estándar | Mide cuánto se dispersan los saldos alrededor de la media. Un valor mayor indica más dispersión; no significa que todos los saldos estén a esa distancia de la media. |
| Mínimo | Menor saldo conocido que aparece en el archivo. |
| Percentil 25 | Valor que deja aproximadamente el 25 % de los saldos en él o por debajo, después de ordenarlos de menor a mayor. |
| Mediana | Valor central de los saldos ordenados; corresponde al percentil 50. |
| Percentil 75 | Valor que deja aproximadamente el 75 % de los saldos en él o por debajo. |
| Percentil 95 | Valor que deja aproximadamente el 95 % de los saldos en él o por debajo. Me ayuda a reconocer niveles altos dentro del archivo. |
| Percentil 99 | Valor que deja aproximadamente el 99 % de los saldos en él o por debajo. Describe la parte más alta de la distribución. |
| Máximo | Mayor saldo conocido que aparece en el archivo. |

Para entender los percentiles, imagino todos los saldos **ordenados de menor a mayor** y ubico el punto que corresponde al porcentaje indicado. Los porcentajes se refieren a observaciones del saldo, no a productos, tiendas o unidades vendidas.

### Cómo leo las otras dos tablas

En la tabla por categoría, `Productos` cuenta SKU ERP distintos y `Snapshots` cuenta filas. `Saldos_conocidos` cuenta filas con cantidad numérica, `Saldos_desconocidos` cuenta filas sin cantidad interpretable y `Ceros` cuenta filas que informan cero. `Mediana_saldo` y `Maximo_saldo` describen únicamente los saldos conocidos de esa categoría. Un producto puede aportar muchas filas porque aparece en varias tiendas y fechas.

La última tabla localiza el máximo: `fecha` indica el día observado, `tienda_id` la tienda, `sku_erp` el producto y `cantidad_en_stock` el saldo original. Encontrar el máximo me permite revisar un caso extremo; no demuestra un error de captura.

### Para qué me sirve y qué no puedo concluir todavía

Uso estas tablas para conocer los niveles habituales y extremos del archivo y ver diferencias entre categorías. Estas estadísticas aisladas no clasifican el inventario como escaso o excesivo: no incorporan demanda ni unidades equivalentes entre presentaciones. No aplico un único límite de error a todos los productos.

**Este resumen no representa el inventario total disponible en un momento.** Por ejemplo, si ayer un producto tenía diez unidades en una tienda y hoy tiene doce, son dos observaciones de su saldo. Sumarlas como veintidós unidades disponibles contaría existencias de momentos diferentes. Este ejemplo es ilustrativo, no una selección de filas del archivo. Tampoco interpreto la diferencia entre dos saldos como ventas: podrían intervenir compras, devoluciones u otros movimientos que este JSON no contiene.

In [6]:
# Describo saldos conocidos y categorías sin confundir frecuencias de snapshots con unidades vendidas.
descripcion = snapshots["stock_num"].describe(percentiles=[.25,.5,.75,.95,.99]).rename(index={"count": "Observaciones", "mean": "Media", "std": "Desviación estándar", "min": "Mínimo", "max": "Máximo", "25%": "Percentil 25", "50%": "Mediana", "75%": "Percentil 75", "95%": "Percentil 95", "99%": "Percentil 99"}).rename_axis("Estadístico").reset_index(name="Saldo observado")
mostrar("distribucion", descripcion, "Todos los saldos numéricos conocidos: Observaciones cuenta snapshots; la media no se pondera, la desviación es muestral y los percentiles describen la posición del saldo en la distribución. El archivo no explicita la unidad física del saldo.")
categorias = snapshots.merge(productos[["sku_erp", "categoria"]], on="sku_erp", validate="many_to_one").groupby("categoria").agg(Productos=("sku_erp", "nunique"), Snapshots=("fecha", "size"), Saldos_conocidos=("stock_num", "count"), Saldos_desconocidos=("stock_num", lambda s: s.isna().sum()), Ceros=("stock_num", lambda s: s.eq(0).sum()), Mediana_saldo=("stock_num", "median"), Maximo_saldo=("stock_num", "max")).reset_index()
mostrar("categorias", categorias, "Todas las categorías del catálogo vinculadas internamente por ERP; mediana y máximo usan solo saldos conocidos. Productos cuenta SKU distintos, el resto de conteos son filas; no comparo unidades económicas entre categorías.")
extremos = snapshots.loc[snapshots["stock_num"].eq(snapshots["stock_num"].max()), ["fecha", "tienda_id", "sku_erp", "cantidad_en_stock"]].sort_values(["fecha", "tienda_id", "sku_erp"]).head(10)
mostrar("extremos", extremos, "Hasta diez snapshots empatados en el máximo global numérico; ordeno fecha, tienda y SKU de forma ascendente y conservo todos los empates si no superan diez. Un máximo es una señal descriptiva, no un error confirmado.")

**Leyenda:** Todos los saldos numéricos conocidos: Observaciones cuenta snapshots; la media no se pondera, la desviación es muestral y los percentiles describen la posición del saldo en la distribución. El archivo no explicita la unidad física del saldo. No muestro el índice de pandas: no es una clave de origen.

Estadístico,Saldo observado
Observaciones,226359.000000
Media,39.817675
Desviación estándar,26.038736
Mínimo,0.000000
Percentil 25,18.000000
Mediana,37.000000
Percentil 75,58.000000
Percentil 95,86.000000
Percentil 99,104.000000
Máximo,153.000000


**Leyenda:** Todas las categorías del catálogo vinculadas internamente por ERP; mediana y máximo usan solo saldos conocidos. Productos cuenta SKU distintos, el resto de conteos son filas; no comparo unidades económicas entre categorías. No muestro el índice de pandas: no es una clave de origen.

categoria,Productos,Snapshots,Saldos_conocidos,Saldos_desconocidos,Ceros,Mediana_saldo,Maximo_saldo
bebidas,8,29848,29278,570,1532,37.0,153.0
cafe_grano,12,38038,37281,757,1911,38.0,143.0
cafe_molido,17,55692,54658,1034,2822,37.0,146.0
comida_caliente,11,34580,33918,662,1747,37.0,142.0
mercancia,15,49140,48202,938,2481,37.0,146.0
panaderia,7,23478,23022,456,1233,37.0,138.0


**Leyenda:** Hasta diez snapshots empatados en el máximo global numérico; ordeno fecha, tienda y SKU de forma ascendente y conservo todos los empates si no superan diez. Un máximo es una señal descriptiva, no un error confirmado. No muestro el índice de pandas: no es una clave de origen.

fecha,tienda_id,sku_erp,cantidad_en_stock
2026-03-28,T004,ERP-PROV-MX-049-A,153


## 6. Rachas de ceros observados

Identifico rachas por tienda–SKU con fechas diarias consecutivas y saldo exactamente cero. `N/A`, una fecha ausente o un saldo distinto de cero interrumpen la evidencia de continuidad; no prueban recuperación del stock. Muestro las secuencias de más de tres días que intersectan el último trimestre calendario completo disponible. La longitud se calcula con toda la ventana antes de recortarla al trimestre para conservar contexto. Un saldo diario cero no demuestra desabasto durante las 24 horas; faltan hora de corte y definición de disponible. Los extremos del archivo pueden censurar el inicio o fin real de una racha.

In [7]:
# Construyo rachas continuas sin atravesar desconocidos y comparo toda la ventana con el trimestre final.
ordenado = snapshots.sort_values(["tienda_id", "sku_erp", "fecha_dt"]).copy()
grupo = ordenado.groupby(["tienda_id", "sku_erp"])
ordenado["cero"] = ordenado["stock_num"].eq(0)
continua = ordenado["cero"] & grupo["stock_num"].shift().eq(0) & grupo["fecha_dt"].diff().eq(pd.Timedelta(days=1))
ordenado["racha_id"] = (~continua).cumsum()
rachas = ordenado.loc[ordenado["cero"]].groupby(["tienda_id", "sku_erp", "racha_id"]).agg(Inicio=("fecha_dt", "min"), Fin=("fecha_dt", "max"), Dias_cero=("fecha_dt", "size")).reset_index().drop(columns="racha_id")
trimestre = fin.to_period("Q")
inicio_trimestre, fin_trimestre = trimestre.start_time.normalize(), trimestre.end_time.normalize()
en_trimestre = rachas.loc[rachas["Fin"].ge(inicio_trimestre) & rachas["Inicio"].le(fin_trimestre)].copy()
en_trimestre["Dias_cero_en_trimestre"] = (en_trimestre["Fin"].clip(upper=fin_trimestre)-en_trimestre["Inicio"].clip(lower=inicio_trimestre)).dt.days+1
largas = en_trimestre.loc[en_trimestre["Dias_cero_en_trimestre"].gt(3)].sort_values(["tienda_id", "sku_erp", "Inicio"])
mostrar("resumen_rachas", medidas({"Inicio trimestre de referencia": str(inicio_trimestre.date()), "Fin trimestre de referencia": str(fin_trimestre.date()), "Rachas cero en toda la ventana": len(rachas), "Mayor racha cero en toda la ventana (días)": int(rachas["Dias_cero"].max()), "Rachas de más de tres días en toda la ventana": int(rachas["Dias_cero"].gt(3).sum()), "Rachas de más de tres días dentro del trimestre": len(largas), "Tiendas con esas rachas trimestrales": largas["tienda_id"].nunique(), "Rachas que tocan primera fecha del archivo": int(rachas["Inicio"].eq(inicio).sum()), "Rachas que tocan última fecha del archivo": int(rachas["Fin"].eq(fin).sum())}), "Cuento secuencias y tiendas por separado; el trimestre deriva del máximo snapshot, no de la fecha de ejecución. Las rachas en bordes tienen duración observada mínima; no afirmo quiebre operativo confirmado.")
mostrar("rachas_largas", largas.assign(Inicio=largas["Inicio"].dt.strftime("%Y-%m-%d"), Fin=largas["Fin"].dt.strftime("%Y-%m-%d")), "Todas las rachas con al menos cuatro días cero dentro del trimestre, sin selección top; Dias_cero mide toda la secuencia observada y Dias_cero_en_trimestre su intersección trimestral. Las fechas son inclusivas.")

**Leyenda:** Cuento secuencias y tiendas por separado; el trimestre deriva del máximo snapshot, no de la fecha de ejecución. Las rachas en bordes tienen duración observada mínima; no afirmo quiebre operativo confirmado. No muestro el índice de pandas: no es una clave de origen.

Medida,Resultado
Inicio trimestre de referencia,2026-01-01
Fin trimestre de referencia,2026-03-31
Rachas cero en toda la ventana,11094
Mayor racha cero en toda la ventana (días),4
Rachas de más de tres días en toda la ventana,3
Rachas de más de tres días dentro del trimestre,3
Tiendas con esas rachas trimestrales,3
Rachas que tocan primera fecha del archivo,60
Rachas que tocan última fecha del archivo,61


**Leyenda:** Todas las rachas con al menos cuatro días cero dentro del trimestre, sin selección top; Dias_cero mide toda la secuencia observada y Dias_cero_en_trimestre su intersección trimestral. Las fechas son inclusivas. No muestro el índice de pandas: no es una clave de origen.

tienda_id,sku_erp,Inicio,Fin,Dias_cero,Dias_cero_en_trimestre
T015,ERP-PROV-MX-014-D,2026-02-09,2026-02-12,4,4
T023,ERP-PROV-MX-046-A,2026-01-25,2026-01-28,4,4
T038,ERP-PROV-MX-040-A,2026-03-18,2026-03-21,4,4


## 7. Equivalencias y catálogo

Reviso cardinalidades y cobertura interna antes de unir otras fuentes. Excluyo null de las pruebas de unicidad de claves; no enlazo productos por nombre ni por sufijo. El inventario no permite contar qué SKU o handles faltan realmente en los otros sistemas porque no los consulto. El archivo no incorpora vigencias de mapeos y los nombres compartidos no identifican variantes de forma única.

In [8]:
# Reviso cardinalidades y referencias internas sin inventar equivalencias para claves incompletas.
perfil_mapeos = pd.DataFrame([{"Clave": c, "Filas": len(mapeos), "Null literal": int(mapeos[c].isna().sum()), "Valores no nulos distintos": mapeos[c].nunique(), "Duplicados adicionales no nulos": int(mapeos[c].dropna().duplicated().sum())} for c in mapeos])
mostrar("mapeos", perfil_mapeos, "Todas las equivalencias declaradas; los valores null no participan en unicidad ni permiten unir registros. No se compara con claves presentes en sales o e-commerce.")
completitud = mapeos.assign(ERP_informado=mapeos["sku_erp"].notna(), Handle_informado=mapeos["handle"].notna()).groupby(["ERP_informado", "Handle_informado"]).size().reset_index(name="Equivalencias")
mostrar("completitud", completitud, "Todas las combinaciones observadas de presencia ERP y handle: True significa informado y False significa null literal. Equivalencias cuenta filas del mapeo, no ventas.")
sin_erp = mapeos.loc[mapeos["sku_erp"].isna()].sort_values("sku_pos")
mostrar("sin_erp", sin_erp.fillna("<null>"), "Todas las filas de mapeo con ERP null; <null> es solo representación visible de un nulo original, no clave creada.")
sin_mapeo = productos.loc[~productos["sku_erp"].isin(mapeos["sku_erp"].dropna()), ["sku_erp", "nombre", "categoria"]].sort_values("sku_erp")
mostrar("catalogo_sin_mapeo", sin_mapeo, "Todos los productos ERP sin referencia no nula en sku_mappings; no significa que falten en POS ni que no se vendan online.")
nombres = productos.groupby("nombre").agg(Productos_ERP=("sku_erp", "nunique"), Categorias=("categoria", "nunique")).reset_index()
mostrar("nombres", medidas({"Productos del catálogo": len(productos), "Categorías": productos["categoria"].nunique(), "Nombres distintos": productos["nombre"].nunique(), "Nombres usados por más de un ERP": int(nombres["Productos_ERP"].gt(1).sum()), "Productos con un nombre compartido": int(productos["nombre"].duplicated(keep=False).sum()), "Productos sin snapshots en esta extracción": int((~productos["sku_erp"].isin(snapshots["sku_erp"])).sum())}), "Cuento nombres exactos y productos afectados por separado; un nombre compartido no permite deduplicar ni reconstruir equivalencias comerciales.")

**Leyenda:** Todas las equivalencias declaradas; los valores null no participan en unicidad ni permiten unir registros. No se compara con claves presentes en sales o e-commerce. No muestro el índice de pandas: no es una clave de origen.

Clave,Filas,Null literal,Valores no nulos distintos,Duplicados adicionales no nulos
sku_pos,65,0,65,0
sku_erp,65,5,60,0
handle,65,38,27,0


**Leyenda:** Todas las combinaciones observadas de presencia ERP y handle: True significa informado y False significa null literal. Equivalencias cuenta filas del mapeo, no ventas. No muestro el índice de pandas: no es una clave de origen.

ERP_informado,Handle_informado,Equivalencias
False,False,1
False,True,4
True,False,37
True,True,23


**Leyenda:** Todas las filas de mapeo con ERP null; <null> es solo representación visible de un nulo original, no clave creada. No muestro el índice de pandas: no es una clave de origen.

sku_pos,sku_erp,handle
CN-00006,<null>,estándar-cafe-grano-006
CN-00016,<null>,<null>
CN-00026,<null>,premium-cafe-molido-026
CN-00036,<null>,gourmet-cafe-grano-036
CN-00046,<null>,termo-mercancia-046


**Leyenda:** Todos los productos ERP sin referencia no nula en sku_mappings; no significa que falten en POS ni que no se vendan online. No muestro el índice de pandas: no es una clave de origen.

sku_erp,nombre,categoria
ERP-PROV-MX-001-A,Sándwich Comida Caliente,comida_caliente
ERP-PROV-MX-006-C,Estándar Cafe Grano,cafe_grano
ERP-PROV-MX-011-C,Bolillo Panaderia,panaderia
ERP-PROV-MX-016-C,Quiche Comida Caliente,comida_caliente
ERP-PROV-MX-021-A,Concha Panaderia,panaderia
ERP-PROV-MX-026-A,Premium Cafe Molido,cafe_molido
ERP-PROV-MX-031-A,Termo Mercancia,mercancia
ERP-PROV-MX-036-A,Gourmet Cafe Grano,cafe_grano
ERP-PROV-MX-041-D,Gourmet Cafe Molido,cafe_molido
ERP-PROV-MX-046-A,Termo Mercancia,mercancia


**Leyenda:** Cuento nombres exactos y productos afectados por separado; un nombre compartido no permite deduplicar ni reconstruir equivalencias comerciales. No muestro el índice de pandas: no es una clave de origen.

Medida,Resultado
Productos del catálogo,70
Categorías,6
Nombres distintos,36
Nombres usados por más de un ERP,23
Productos con un nombre compartido,57
Productos sin snapshots en esta extracción,0


## 8. Historial y vigencias de costos

Conservo fecha, costo y proveedor originales; añado fecha interpretada y costo numérico. Ordeno cronológicamente para comparar vigencias aunque el JSON tenga otro orden. Evalúo como escenario la última vigencia no posterior al día del snapshot, sin adoptarla como regla aprobada. Una cobertura temporal completa no valida unidad, impuestos, gastos incluidos, política de costo ni aplicación intradía.

In [9]:
# Valido historiales y cambios de costo como evidencia temporal sin calcular márgenes ni cruzar ventas.
costos["vigencia_dt"] = pd.to_datetime(costos["fecha_vigencia"], format="%Y-%m-%d", errors="coerce")
costos["costo_num"] = pd.to_numeric(costos["costo_mxn"], errors="coerce")
primera_vigencia = costos.groupby("sku_erp")["vigencia_dt"].min()
primera_snapshot = snapshots.groupby("sku_erp")["fecha_dt"].min()
historiales_desordenados = sum(not pd.Series([c["fecha_vigencia"] for c in p["cost_history"]]).is_monotonic_increasing for p in datos["catalogo"]["productos"])
control_costos = {"Entradas de costo": len(costos), "Productos con historial no vacío": costos["sku_erp"].nunique(), "Primera vigencia": str(costos["vigencia_dt"].min().date()), "Última vigencia": str(costos["vigencia_dt"].max().date()), "Fechas inválidas": int(costos["vigencia_dt"].isna().sum()), "Fechas fuera de formato ISO diario": int((~costos["fecha_vigencia"].str.fullmatch(r"[0-9]{4}-[0-9]{2}-[0-9]{2}")).sum()), "Claves SKU–vigencia adicionales": int(costos.duplicated(["sku_erp", "fecha_vigencia"]).sum()), "Costos no numéricos o no finitos": int((~np.isfinite(costos["costo_num"])).sum()), "Costos cero": int(costos["costo_num"].eq(0).sum()), "Costos negativos": int(costos["costo_num"].lt(0).sum()), "Historiales no ordenados en origen": historiales_desordenados, "Vigencias posteriores al último snapshot": int(costos["vigencia_dt"].gt(fin).sum()), "Productos observados sin costo desde su primer snapshot": int((primera_snapshot < primera_vigencia.reindex(primera_snapshot.index)).sum() + primera_vigencia.reindex(primera_snapshot.index).isna().sum())}
mostrar("costos", medidas(control_costos), "Todos los historiales; entradas cuenta versiones de costo y claves adicionales excluye la primera. La cobertura es potencial bajo última vigencia anterior o igual, sin afirmar método contable ni costo por unidad confirmado.")
costos_ordenados = costos.sort_values(["sku_erp", "vigencia_dt"]).copy()
costos_ordenados["costo_previo"] = costos_ordenados.groupby("sku_erp")["costo_num"].shift()
costos_ordenados["cambio_mxn"] = costos_ordenados["costo_num"]-costos_ordenados["costo_previo"]
costos_ordenados["cambio_pct"] = 100*costos_ordenados["cambio_mxn"]/costos_ordenados["costo_previo"]
mostrar("distribucion_costos", costos["costo_num"].describe(percentiles=[.25,.5,.75,.95]).rename(index={"count": "Observaciones", "mean": "Media", "std": "Desviación estándar", "min": "Mínimo", "max": "Máximo", "25%": "Percentil 25", "50%": "Mediana", "75%": "Percentil 75", "95%": "Percentil 95"}).rename_axis("Estadístico").reset_index(name="Costo registrado"), "Todas las versiones de costo; Observaciones cuenta vigencias y el resto expresa MXN por la unidad de costo no explicitada en el archivo. Media y percentiles no se ponderan por tiempo, compras ni ventas y no representan el costo vigente promedio.")
mostrar("cambios_costos", medidas({"Cambios comparables": int(costos_ordenados["costo_previo"].notna().sum()), "Aumentos": int(costos_ordenados["cambio_mxn"].gt(0).sum()), "Bajadas": int(costos_ordenados["cambio_mxn"].lt(0).sum()), "Sin cambio": int(costos_ordenados["cambio_mxn"].eq(0).sum()), "Mayor subida porcentual": round(costos_ordenados["cambio_pct"].max(), 4), "Mayor bajada porcentual": round(costos_ordenados["cambio_pct"].min(), 4)}), "Comparo cada costo con la vigencia previa del mismo ERP, sin ponderar por compras; las primeras entradas no tienen cambio calculable. Los extremos porcentuales son señales y no errores demostrados.")
mostrar("extremos_costos", costos_ordenados.loc[costos_ordenados["cambio_pct"].eq(costos_ordenados["cambio_pct"].max()) | costos_ordenados["cambio_pct"].eq(costos_ordenados["cambio_pct"].min()), ["sku_erp", "fecha_vigencia", "costo_previo", "costo_mxn", "cambio_mxn", "cambio_pct"]], "Incluyo todos los empates de mayor subida y bajada porcentual de costo; cambio_mxn está en MXN por la unidad de costo no explicitada en el archivo, cambio_pct es porcentaje sobre el costo anterior. No son márgenes.")

**Leyenda:** Todos los historiales; entradas cuenta versiones de costo y claves adicionales excluye la primera. La cobertura es potencial bajo última vigencia anterior o igual, sin afirmar método contable ni costo por unidad confirmado. No muestro el índice de pandas: no es una clave de origen.

Medida,Resultado
Entradas de costo,282
Productos con historial no vacío,70
Primera vigencia,2024-10-01
Última vigencia,2026-03-30
Fechas inválidas,0
Fechas fuera de formato ISO diario,0
Claves SKU–vigencia adicionales,0
Costos no numéricos o no finitos,0
Costos cero,0
Costos negativos,0


**Leyenda:** Todas las versiones de costo; Observaciones cuenta vigencias y el resto expresa MXN por la unidad de costo no explicitada en el archivo. Media y percentiles no se ponderan por tiempo, compras ni ventas y no representan el costo vigente promedio. No muestro el índice de pandas: no es una clave de origen.

Estadístico,Costo registrado
Observaciones,282.000000
Media,127.709752
Desviación estándar,98.912367
Mínimo,7.730000
Percentil 25,45.742500
Mediana,102.115000
Percentil 75,186.600000
Percentil 95,288.003500
Máximo,478.300000


**Leyenda:** Comparo cada costo con la vigencia previa del mismo ERP, sin ponderar por compras; las primeras entradas no tienen cambio calculable. Los extremos porcentuales son señales y no errores demostrados. No muestro el índice de pandas: no es una clave de origen.

Medida,Resultado
Cambios comparables,212
Aumentos,117
Bajadas,95
Sin cambio,0
Mayor subida porcentual,9.8793
Mayor bajada porcentual,-7.9589


**Leyenda:** Incluyo todos los empates de mayor subida y bajada porcentual de costo; cambio_mxn está en MXN por la unidad de costo no explicitada en el archivo, cambio_pct es porcentaje sobre el costo anterior. No son márgenes. No muestro el índice de pandas: no es una clave de origen.

sku_erp,fecha_vigencia,costo_previo,costo_mxn,cambio_mxn,cambio_pct
ERP-PROV-MX-006-C,2025-05-28,110.82,102.00,-8.82,-7.958852
ERP-PROV-MX-011-C,2025-06-20,13.26,14.57,1.31,9.879336


## 9. Validaciones independientes y conservación

Contrasto los recuentos y el calendario con `Counter`, `date` y `timedelta`, sin depender de las agrupaciones de pandas. Reconstruyo las rachas directamente a partir del JSON. Estas validaciones apoyan la coherencia del análisis, no confirman integridad del ERP ni reglas del negocio. Compruebo al terminar que los bytes y las columnas originales permanecen intactos.

In [10]:
# Contrasto calendario y rachas por un segundo método y verifico conservación exacta de la fuente.
filas = datos["snapshots"]
conteos_clave = Counter((r["tienda_id"], r["sku_erp"], r["fecha"]) for r in filas)
por_par = {}
for r in filas:
    por_par.setdefault((r["tienda_id"], r["sku_erp"]), {})[date.fromisoformat(r["fecha"])] = r["cantidad_en_stock"]
calendario_std = {inicio.date()+timedelta(days=i) for i in range((fin.date()-inicio.date()).days+1)}
rachas_std = []
for (tienda, sku), diario in sorted(por_par.items()):
    comienzo, anterior = None, None
    for dia, valor in sorted(diario.items()):
        cero = isinstance(valor, (int, float)) and not isinstance(valor, bool) and valor == 0
        if comienzo is not None and (not cero or dia != anterior+timedelta(days=1)):
            rachas_std.append((tienda, sku, comienzo, anterior, (anterior-comienzo).days+1))
            comienzo = None
        if cero and comienzo is None: comienzo = dia
        anterior = dia
    if comienzo is not None: rachas_std.append((tienda, sku, comienzo, anterior, (anterior-comienzo).days+1))
rachas_pd = [(r.tienda_id, r.sku_erp, r.Inicio.date(), r.Fin.date(), r.Dias_cero) for r in rachas.itertuples()]
assert sorted(rachas_std) == sorted(rachas_pd)
assert len(conteos_clave) == len(filas)
assert all(set(v) == calendario_std for v in por_par.values())
assert sum(r["cantidad_en_stock"] == "N/A" for r in filas) == int(snapshots["stock_num"].isna().sum())
assert sum(isinstance(r["cantidad_en_stock"], int) and r["cantidad_en_stock"] == 0 for r in filas) == int(snapshots["stock_num"].eq(0).sum())
assert len({(c["sku_erp"], date.fromisoformat(c["fecha_vigencia"])) for c in costos_filas}) == len(costos_filas)
assert all(Decimal(str(c["costo_mxn"])).is_finite() and Decimal(str(c["costo_mxn"])) > 0 for c in costos_filas)
assert all(min(date.fromisoformat(c["fecha_vigencia"]) for c in p["cost_history"]) <= inicio.date() for p in datos["catalogo"]["productos"])
assert datos == original
pd.testing.assert_frame_equal(snapshots_original, snapshots[snapshots_original.columns])
assert hashlib.sha256(ruta.read_bytes()).hexdigest() == huella
mostrar("validacion", medidas({"Recuento independiente de claves únicas": len(conteos_clave), "Pares contrastados con calendario estándar": len(por_par), "Rachas contrastadas con método independiente": len(rachas_std), "Columnas originales conservadas": True, "JSON en memoria conservado": True, "SHA-256 final coincide": True}), "Controles ejecutados con assert; si falla alguno detengo la ejecución. True expresa resultado técnico de la IA, no aprobación personal ni validación contra el ERP.")

**Leyenda:** Controles ejecutados con assert; si falla alguno detengo la ejecución. True expresa resultado técnico de la IA, no aprobación personal ni validación contra el ERP. No muestro el índice de pandas: no es una clave de origen.

Medida,Resultado
Recuento independiente de claves únicas,230776
Pares contrastados con calendario estándar,1268
Rachas contrastadas con método independiente,11094
Columnas originales conservadas,True
JSON en memoria conservado,True
SHA-256 final coincide,True


## Conclusión

Tengo caracterizada la extracción de inventario y sus controles internos: separo saldos desconocidos, ceros y ausencias, y describo rachas observadas, equivalencias y costos. Este diagnóstico aislado no demuestra quiebres intradía ni calcula rotación o margen.